# PEGASUS pretrained on CNN/DailyMail\n## M.Tech Dissertation — A Two-Stage Summarisation Pipeline\n\n**Model:** `google/pegasus-cnn_dailymail`  \n**Dataset:** CNN/DailyMail 3.0.0  \n**Training samples:** none (already fine-tuned on CNN/DailyMail)  \n**GPU required:** T4 15 GB VRAM (for fast inference and evaluation)\n\n> **Before running:** `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
# ── Step 1: Verify GPU ──────────────────────────────────────────────────────
# If assertion fails: Runtime → Change runtime type → Hardware accelerator → T4 GPU
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU then re-run."
print("GPU  :", torch.cuda.get_device_name(0))
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("PyTorch:", torch.__version__)
print("OK — GPU ready.")


In [ ]:
# ── Step 2: Install packages (mirrors requirements.txt) ─────────────────────
import subprocess, sys
pkgs = [
    "transformers>=4.40.0", "datasets>=2.18.0", "accelerate>=1.1.0",
    "sentencepiece>=0.1.99", "sentence-transformers>=2.7.0", "rank_bm25>=0.2.2",
    "nltk>=3.8.1", "rouge-score>=0.1.2", "bert-score>=0.3.13", "evaluate>=0.4.1",
    "sacrebleu>=2.4.0", "matplotlib>=3.8.0", "pandas>=2.1.0", "numpy>=1.26.0",
    "tqdm>=4.66.0", "scikit-learn>=1.4.0", "scipy>=1.12.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("All packages installed.")


In [ ]:
# ── Step 4: NLTK tokeniser data ─────────────────────────────────────────────
import nltk
nltk.download("punkt_tab", quiet=True)   # NLTK 3.8+
nltk.download("punkt",     quiet=True)   # fallback
print("NLTK data ready.")


In [ ]:
# ── config.py — inline copy (same as dissertation/config.py) ──────────────────
# Hyperparameters are kept identical to the codebase for reproducibility.
import os
from pathlib import Path

ROOT        = Path("/content")
DATA_DIR    = ROOT / "data" / "cache"
CKPT_DIR    = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_NAME    = "cnn_dailymail"
DATASET_VERSION = "3.0.0"
ARTICLE_COL     = "article"
SUMMARY_COL     = "highlights"
SEED            = 42

# 50,000 training examples: ~17% of CNN/DM, achievable in one T4 session (~1.5h)
# Justified in dissertation Section 3.x: resource-constrained fine-tuning
MAX_TRAIN_SAMPLES = 12000

BART_MAX_INPUT     = 1024
BART_MAX_OUTPUT    = 128
PEGASUS_MAX_INPUT  = 512
PEGASUS_MAX_OUTPUT = 128

BART_MODEL    = "facebook/bart-large-cnn"
PEGASUS_MODEL = "google/pegasus-cnn_dailymail"  # used directly; no project checkpoint
MBART_MODEL   = "facebook/mbart-large-cc25"

BART_CKPT    = CKPT_DIR / "bart_finetuned"
MBART_CKPT   = CKPT_DIR / "mbart_finetuned"

ROUGE_TYPES   = ["rouge1", "rouge2", "rougeL"]


In [ ]:
# ── data/data_pipeline.py — inline copy (same as dissertation codebase) ───────
import re, logging
import numpy as np
from datasets import load_dataset, DatasetDict
import nltk
from nltk.tokenize import sent_tokenize

log = logging.getLogger(__name__)

def clean_text(text):
    """Remove CNN/DM boilerplate and normalise whitespace."""
    text = re.sub(r"^\s*\(CNN\)\s*[-–—]?\s*", "", text)
    text = re.sub(r"\(By\s+[^)]+\)", "", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

def preprocess_example(example):
    """Clean article and summary; add sentence count."""
    article = clean_text(example[ARTICLE_COL])
    summary = clean_text(example[SUMMARY_COL])
    summary = summary.replace("\n", " ")
    sents   = sent_tokenize(article)
    return {
        ARTICLE_COL:     article,
        SUMMARY_COL:     summary,
        "article_sents": sents,
        "num_sents":     len(sents),
        "article_len":   len(article.split()),
        "summary_len":   len(summary.split()),
    }

def get_datasets(force_reprocess=False):
    """Load, clean, and split CNN/DailyMail. Cached after first run."""
    cache_path = DATA_DIR / "cnn_dm_processed"
    if cache_path.exists() and not force_reprocess:
        from datasets import load_from_disk
        return load_from_disk(str(cache_path))
    raw       = load_dataset(DATASET_NAME, DATASET_VERSION)
    processed = raw.map(preprocess_example, batched=False, desc="Preprocessing")
    train = processed["train"]
    val   = processed["validation"]
    test  = processed["test"]
    if MAX_TRAIN_SAMPLES is not None:
        train = train.shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))
    splits = DatasetDict({"train": train, "validation": val, "test": test})
    splits.save_to_disk(str(cache_path))
    return splits


## Direct inference\nPEGASUS is already trained on CNN/DailyMail, so no project fine-tuning step is required.

In [ ]:
# ── Direct PEGASUS evaluation on test split ─────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import evaluate as hf_evaluate, json

tokenizer_eval = AutoTokenizer.from_pretrained(PEGASUS_MODEL)
model_eval     = AutoModelForSeq2SeqLM.from_pretrained(PEGASUS_MODEL).cuda()
rouge          = hf_evaluate.load("rouge")
splits         = get_datasets()
test_articles  = splits["test"][ARTICLE_COL][:500]
test_summaries = splits["test"][SUMMARY_COL][:500]

summariser = pipeline("summarization", model=model_eval, tokenizer=tokenizer_eval,
                      device=0, max_length=128, min_length=20, truncation=True)
preds  = [r[0]["summary_text"] for r in summariser(test_articles, batch_size=4)]
scores = rouge.compute(predictions=preds, references=test_summaries, use_stemmer=True)
scores = {k: round(v * 100, 2) for k, v in scores.items()}
print("Test ROUGE:", scores)
out = RESULTS_DIR / "test_rouge_pegasus.json"
with open(out, "w") as f: json.dump(scores, f, indent=2)
print("Saved to", out)

## Next steps\n1. No checkpoint download is needed\n2. Run `python experiments/run_pipeline.py --max_samples 500` to compare it with BART and mBART